# PRMP vs Standard HeteroSAGE on RelBench rel-avito ad-ctr (Pure PyTorch)

This notebook demonstrates **Predictive Residual Message Passing (PRMP)** — a novel GNN message-passing variant where parent nodes predict child features and aggregate *residuals* instead of raw messages.

**What this experiment does:**
- Trains 3 GNN variants (StandardSAGE, PRMP, WideSAGE parameter-matched control) on the rel-avito ad-ctr regression task
- Uses pure PyTorch (no torch-geometric) with a heterogeneous graph of 8 tables and 11 FK links
- Compares RMSE, embedding-space R², and PRMP prediction MLP R² across models
- Performs Cohen's d effect-size analysis and cardinality×R² regime analysis

**Key finding:** PRMP achieves mean RMSE 0.0878±0.0005, slightly outperforming StandardSAGE (0.0882±0.0009) with Cohen's d=0.47.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.3', 'psutil==7.0.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'torch==2.9.0+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import copy
import gc
import itertools
import json
import math
import os
import sys
import time
from hashlib import md5

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter6_prmp_vs_standar/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Models: {data['metadata']['models']}")
print(f"FK links in r2_diagnostic: {len(data['metadata']['r2_diagnostic'])}")

## Configuration

All tunable parameters for the demo. We use small values so the notebook runs quickly on a synthetic graph.

In [ ]:
# --- Tunable parameters ---
# Original values in comments; demo uses small values for speed.

HIDDEN_DIM = 16          # Original: 128
NUM_LAYERS = 2           # Original: 2
DROPOUT = 0.3            # Original: 0.3
LR = 0.001               # Original: 0.001
WEIGHT_DECAY = 1e-5      # Original: 1e-5
MAX_EPOCHS = 5           # Original: 200
PATIENCE = 3             # Original: 20
SEEDS = [42]             # Original: [42, 123, 456]
TEXT_HASH_DIM = 4        # Original: 16

# Synthetic graph sizes for demo
N_ADS = 100              # Number of AdsInfo nodes
N_CATEGORY = 10          # Number of Category nodes
N_LOCATION = 20          # Number of Location nodes
N_EDGES_PER_LINK = 150   # Edges per FK link

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Phase 2: Pure PyTorch GNN Layers

Core building blocks: `scatter_mean` aggregation, standard `BipartiteSAGEConv`, and the novel `PRMPSAGEConv` which predicts child features from parent embeddings and aggregates the *residuals*.

In [ ]:
def scatter_mean(src: torch.Tensor, index: torch.Tensor, dim_size: int) -> torch.Tensor:
    """Scatter mean aggregation: aggregate src by index."""
    out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
    count = torch.zeros(dim_size, 1, device=src.device, dtype=src.dtype)
    out.scatter_add_(0, index.unsqueeze(1).expand_as(src), src)
    ones = torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype)
    count.scatter_add_(0, index.unsqueeze(1), ones)
    count = count.clamp(min=1)
    return out / count


class BipartiteSAGEConv(nn.Module):
    """SAGEConv for bipartite edges: aggregates src features to dst nodes."""
    def __init__(self, src_dim: int, dst_dim: int, out_dim: int):
        super().__init__()
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        neigh_feats = x_src[edge_src]
        agg = scatter_mean(self.lin_neigh(neigh_feats), edge_dst, num_dst)
        out = agg + self.lin_self(x_dst)
        return out


class PRMPSAGEConv(nn.Module):
    """PRMP: parent predicts child features, aggregate residuals."""
    def __init__(self, src_dim: int, dst_dim: int, out_dim: int):
        super().__init__()
        self.pred_mlp = nn.Sequential(
            nn.Linear(dst_dim, src_dim),
            nn.ReLU(),
            nn.Linear(src_dim, src_dim),
        )
        # Initialize last layer near-zero so residuals ~ raw features at start
        nn.init.zeros_(self.pred_mlp[2].weight)
        nn.init.zeros_(self.pred_mlp[2].bias)

        self.residual_norm = nn.LayerNorm(src_dim)
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        # Parent predicts child
        pred_src = self.pred_mlp(x_dst)
        src_feats = x_src[edge_src]
        pred_feats = pred_src[edge_dst]
        residual = self.residual_norm(src_feats - pred_feats)  # THE CORE PRMP OPERATION
        agg = scatter_mean(self.lin_neigh(residual), edge_dst, num_dst)
        out = agg + self.lin_self(x_dst)
        return out

print("GNN layers defined: BipartiteSAGEConv, PRMPSAGEConv")

## Phase 3: Heterogeneous GNN Models

Three model variants: **StandardHeteroSAGE** (baseline), **PRMPHeteroSAGE** (uses PRMP on reverse-FK edges), and **WideHeteroSAGE** (parameter-matched control with wider hidden dim).

In [ ]:
class HeteroGNNLayer(nn.Module):
    """One layer of heterogeneous message passing across all edge types."""
    def __init__(self, conv_dict: nn.ModuleDict):
        super().__init__()
        self.convs = conv_dict

    def forward(self, x_dict, graph, edge_types):
        out_dict = {}
        for etype_key, src_type, dst_type, esrc_key, edst_key, ndst_key in edge_types:
            if etype_key not in self.convs:
                continue
            conv = self.convs[etype_key]
            result = conv(x_dict[src_type], x_dict[dst_type],
                          graph[esrc_key], graph[edst_key], graph[ndst_key])
            if dst_type not in out_dict:
                out_dict[dst_type] = result
            else:
                out_dict[dst_type] = out_dict[dst_type] + result
        return out_dict


class StandardHeteroSAGE(nn.Module):
    """Standard HeteroSAGE with mean aggregation on all edges."""
    def __init__(self, node_dims, edge_types, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS):
        super().__init__()
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_lins = nn.ModuleDict()
        for table_name, feat_dim in node_dims.items():
            self.input_lins[table_name] = nn.Linear(feat_dim, hidden_dim)

        self.layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = nn.ModuleDict()
            for etype_key, src_type, dst_type, *_ in edge_types:
                conv_dict[etype_key] = BipartiteSAGEConv(hidden_dim, hidden_dim, hidden_dim)
            self.layers.append(HeteroGNNLayer(conv_dict))
            norm_dict = nn.ModuleDict()
            for table_name in node_dims:
                norm_dict[table_name] = nn.LayerNorm(hidden_dim)
            self.layer_norms.append(norm_dict)

        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_embeddings(self, graph):
        x_dict = {}
        for table_name, lin in self.input_lins.items():
            feat_key = f"x_{table_name}"
            if feat_key in graph:
                x_dict[table_name] = F.relu(lin(graph[feat_key]))
        for layer_idx in range(self.num_layers):
            out = self.layers[layer_idx](x_dict, graph, self.edge_types)
            norms = self.layer_norms[layer_idx]
            new_x = {}
            for table_name in x_dict:
                if table_name in out:
                    new_x[table_name] = F.relu(norms[table_name](self.dropout(out[table_name])))
                else:
                    new_x[table_name] = x_dict[table_name]
            x_dict = new_x
        return x_dict

    def forward(self, graph, entity_type="AdsInfo"):
        x_dict = self.get_embeddings(graph)
        return self.head(x_dict[entity_type]).squeeze(-1)


class PRMPHeteroSAGE(nn.Module):
    """PRMP HeteroSAGE: uses PRMPSAGEConv on reverse-FK edges."""
    def __init__(self, node_dims, edge_types, prmp_edge_keys, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS):
        super().__init__()
        self.edge_types = edge_types
        self.prmp_edge_keys = prmp_edge_keys
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_lins = nn.ModuleDict()
        for table_name, feat_dim in node_dims.items():
            self.input_lins[table_name] = nn.Linear(feat_dim, hidden_dim)

        self.layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = nn.ModuleDict()
            for etype_key, src_type, dst_type, *_ in edge_types:
                if etype_key in prmp_edge_keys:
                    conv_dict[etype_key] = PRMPSAGEConv(hidden_dim, hidden_dim, hidden_dim)
                else:
                    conv_dict[etype_key] = BipartiteSAGEConv(hidden_dim, hidden_dim, hidden_dim)
            self.layers.append(HeteroGNNLayer(conv_dict))
            norm_dict = nn.ModuleDict()
            for table_name in node_dims:
                norm_dict[table_name] = nn.LayerNorm(hidden_dim)
            self.layer_norms.append(norm_dict)

        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_embeddings(self, graph):
        x_dict = {}
        for table_name, lin in self.input_lins.items():
            feat_key = f"x_{table_name}"
            if feat_key in graph:
                x_dict[table_name] = F.relu(lin(graph[feat_key]))
        for layer_idx in range(self.num_layers):
            out = self.layers[layer_idx](x_dict, graph, self.edge_types)
            norms = self.layer_norms[layer_idx]
            new_x = {}
            for table_name in x_dict:
                if table_name in out:
                    new_x[table_name] = F.relu(norms[table_name](self.dropout(out[table_name])))
                else:
                    new_x[table_name] = x_dict[table_name]
            x_dict = new_x
        return x_dict

    def forward(self, graph, entity_type="AdsInfo"):
        x_dict = self.get_embeddings(graph)
        return self.head(x_dict[entity_type]).squeeze(-1)


class WideHeteroSAGE(nn.Module):
    """Wide HeteroSAGE -- parameter-matched control (wider hidden dim)."""
    def __init__(self, node_dims, edge_types, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS):
        super().__init__()
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_lins = nn.ModuleDict()
        for table_name, feat_dim in node_dims.items():
            self.input_lins[table_name] = nn.Linear(feat_dim, hidden_dim)

        self.layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = nn.ModuleDict()
            for etype_key, src_type, dst_type, *_ in edge_types:
                conv_dict[etype_key] = BipartiteSAGEConv(hidden_dim, hidden_dim, hidden_dim)
            self.layers.append(HeteroGNNLayer(conv_dict))
            norm_dict = nn.ModuleDict()
            for table_name in node_dims:
                norm_dict[table_name] = nn.LayerNorm(hidden_dim)
            self.layer_norms.append(norm_dict)

        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_embeddings(self, graph):
        x_dict = {}
        for table_name, lin in self.input_lins.items():
            feat_key = f"x_{table_name}"
            if feat_key in graph:
                x_dict[table_name] = F.relu(lin(graph[feat_key]))
        for layer_idx in range(self.num_layers):
            out = self.layers[layer_idx](x_dict, graph, self.edge_types)
            norms = self.layer_norms[layer_idx]
            new_x = {}
            for table_name in x_dict:
                if table_name in out:
                    new_x[table_name] = F.relu(norms[table_name](self.dropout(out[table_name])))
                else:
                    new_x[table_name] = x_dict[table_name]
            x_dict = new_x
        return x_dict

    def forward(self, graph, entity_type="AdsInfo"):
        x_dict = self.get_embeddings(graph)
        return self.head(x_dict[entity_type]).squeeze(-1)

print("Model classes defined: StandardHeteroSAGE, PRMPHeteroSAGE, WideHeteroSAGE")

## Phase 4: Build Synthetic Heterogeneous Graph & Train

Since loading the full rel-avito dataset requires ~20GB, we build a small synthetic graph with the same schema (AdsInfo, Category, Location) to demonstrate the models running end-to-end.

In [ ]:
def build_synthetic_graph():
    """Build a small synthetic heterogeneous graph mimicking the avito schema."""
    np.random.seed(42)
    graph = {}

    # Node counts
    graph["n_AdsInfo"] = N_ADS
    graph["n_Category"] = N_CATEGORY
    graph["n_Location"] = N_LOCATION

    # Node features (random, mimicking real feature dims)
    n_ads_feat = 4   # Price, IsContext + 2 text hash dims
    n_cat_feat = 3   # Level, ParentCategoryID, SubcategoryID
    n_loc_feat = 3   # Level, RegionID, CityID

    graph["x_AdsInfo"] = torch.randn(N_ADS, n_ads_feat)
    graph["x_Category"] = torch.randn(N_CATEGORY, n_cat_feat)
    graph["x_Location"] = torch.randn(N_LOCATION, n_loc_feat)

    # FK link definitions for demo: (child, fk_col, parent)
    demo_fk_links = [
        ("AdsInfo", "LocationID", "Location"),
        ("AdsInfo", "CategoryID", "Category"),
    ]

    # Build edges for each FK link
    for child, fk_col, parent in demo_fk_links:
        n_child = graph[f"n_{child}"]
        n_parent = graph[f"n_{parent}"]
        n_edges = min(N_EDGES_PER_LINK, n_child * 3)

        src = torch.randint(0, n_child, (n_edges,))
        dst = torch.randint(0, n_parent, (n_edges,))

        # Forward: child -> parent
        graph[f"edge_{child}_to_{parent}_via_{fk_col}_src"] = src
        graph[f"edge_{child}_to_{parent}_via_{fk_col}_dst"] = dst
        # Reverse: parent -> child
        graph[f"edge_{parent}_to_{child}_via_{fk_col}_src"] = dst
        graph[f"edge_{parent}_to_{child}_via_{fk_col}_dst"] = src

    # Task labels: synthetic regression target for AdsInfo
    graph["y"] = torch.randn(N_ADS).abs() * 0.1  # positive CTR-like values
    n_train = int(N_ADS * 0.7)
    n_val = int(N_ADS * 0.15)
    graph["train_mask"] = torch.zeros(N_ADS, dtype=torch.bool)
    graph["val_mask"] = torch.zeros(N_ADS, dtype=torch.bool)
    graph["test_mask"] = torch.zeros(N_ADS, dtype=torch.bool)
    graph["train_mask"][:n_train] = True
    graph["val_mask"][n_train:n_train+n_val] = True
    graph["test_mask"][n_train+n_val:] = True

    return graph, demo_fk_links

graph, demo_fk_links = build_synthetic_graph()
print(f"Synthetic graph: {graph['n_AdsInfo']} ads, {graph['n_Category']} categories, {graph['n_Location']} locations")
print(f"Labels: train={graph['train_mask'].sum()}, val={graph['val_mask'].sum()}, test={graph['test_mask'].sum()}")

In [ ]:
def discover_edge_types(graph, fk_links):
    """Discover all edge types present in the graph."""
    edge_types = []
    seen = set()
    for child_table, fk_col, parent_table in fk_links:
        # Forward: child -> parent
        src_key = f"edge_{child_table}_to_{parent_table}_via_{fk_col}_src"
        dst_key = f"edge_{child_table}_to_{parent_table}_via_{fk_col}_dst"
        if src_key in graph and dst_key in graph:
            etype = f"{child_table}_to_{parent_table}_via_{fk_col}"
            if etype not in seen:
                edge_types.append((etype, child_table, parent_table, src_key, dst_key, f"n_{parent_table}"))
                seen.add(etype)
        # Reverse: parent -> child
        src_key_r = f"edge_{parent_table}_to_{child_table}_via_{fk_col}_src"
        dst_key_r = f"edge_{parent_table}_to_{child_table}_via_{fk_col}_dst"
        if src_key_r in graph and dst_key_r in graph:
            etype_r = f"{parent_table}_to_{child_table}_via_{fk_col}"
            if etype_r not in seen:
                edge_types.append((etype_r, parent_table, child_table, src_key_r, dst_key_r, f"n_{child_table}"))
                seen.add(etype_r)
    return edge_types


def get_prmp_edge_keys(fk_links):
    """Identify reverse-FK edges (parent->child) where PRMP applies."""
    prmp_keys = set()
    for child_table, fk_col, parent_table in fk_links:
        etype_r = f"{parent_table}_to_{child_table}_via_{fk_col}"
        prmp_keys.add(etype_r)
    return prmp_keys


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}


def solve_wide_hidden_dim(prmp_params, node_dims, edge_types, base_hidden=HIDDEN_DIM):
    """Binary search for hidden_dim that gives WideHeteroSAGE ~ prmp_params."""
    lo, hi = base_hidden, base_hidden * 3
    best_dim = base_hidden
    for _ in range(20):
        mid = (lo + hi) // 2
        model = WideHeteroSAGE(node_dims, edge_types, hidden_dim=mid)
        n_params = sum(p.numel() for p in model.parameters())
        del model
        if abs(n_params - prmp_params) / max(prmp_params, 1) < 0.05:
            best_dim = mid; break
        elif n_params < prmp_params:
            lo = mid + 1
        else:
            hi = mid - 1
        best_dim = mid
    return best_dim


# Discover edge types and setup
edge_types = discover_edge_types(graph, demo_fk_links)
prmp_edge_keys = get_prmp_edge_keys(demo_fk_links)

# Node feature dims
node_dims = {}
for key in graph:
    if key.startswith("x_"):
        table_name = key[2:]
        node_dims[table_name] = graph[key].shape[1]

print(f"Edge types: {len(edge_types)}, PRMP edges: {len(prmp_edge_keys)}")
print(f"Node dims: {node_dims}")

# Parameter counts & wide dim calibration
std_model = StandardHeteroSAGE(node_dims, edge_types)
prmp_model = PRMPHeteroSAGE(node_dims, edge_types, prmp_edge_keys)
std_params = count_parameters(std_model)
prmp_params = count_parameters(prmp_model)
wide_hidden = solve_wide_hidden_dim(prmp_params["total"], node_dims, edge_types)
wide_model = WideHeteroSAGE(node_dims, edge_types, hidden_dim=wide_hidden)
wide_params = count_parameters(wide_model)

print(f"Standard: {std_params['total']:,} params (hidden={HIDDEN_DIM})")
print(f"PRMP:     {prmp_params['total']:,} params (hidden={HIDDEN_DIM})")
print(f"Wide:     {wide_params['total']:,} params (hidden={wide_hidden})")
del std_model, prmp_model, wide_model

In [ ]:
def train_model(model, graph, device, seed, model_name):
    """Train one model instance with early stopping."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    y = graph["y"]
    train_valid = graph["train_mask"] & ~torch.isnan(y)
    val_valid = graph["val_mask"] & ~torch.isnan(y)
    test_valid = graph["test_mask"] & ~torch.isnan(y)

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None
    best_epoch = 0
    t_start = time.time()

    for epoch in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        pred = model(graph, "AdsInfo")
        loss = loss_fn(pred[train_valid], y[train_valid])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_all = model(graph, "AdsInfo")
            val_loss = loss_fn(pred_all[val_valid], y[val_valid]).item() if val_valid.sum() > 0 else loss.item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    elapsed = time.time() - t_start

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_test = model(graph, "AdsInfo")

    if test_valid.sum() > 0:
        test_pred = pred_test[test_valid].cpu().numpy()
        test_true = y[test_valid].cpu().numpy()
        rmse = float(np.sqrt(np.mean((test_pred - test_true) ** 2)))
        mae = float(np.mean(np.abs(test_pred - test_true)))
        ss_res = float(np.sum((test_true - test_pred) ** 2))
        ss_tot = float(np.sum((test_true - test_true.mean()) ** 2))
        r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    else:
        rmse = float(np.sqrt(best_val_loss))
        mae = rmse * 0.8
        r2 = 0.0

    metrics = {"seed": seed, "rmse": round(rmse, 6), "mae": round(mae, 6),
               "r2": round(r2, 6), "best_epoch": best_epoch, "train_time_s": round(elapsed, 1)}
    print(f"  [{model_name}|seed={seed}] RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}, epoch={best_epoch}, time={elapsed:.1f}s")
    return metrics, model

In [ ]:
# Move graph to device
for k, v in graph.items():
    if isinstance(v, torch.Tensor):
        graph[k] = v.to(DEVICE)

# Train all 3 variants
model_configs = [
    ("A_StandardSAGE", lambda: StandardHeteroSAGE(node_dims, edge_types, hidden_dim=HIDDEN_DIM)),
    ("B_PRMP_Full", lambda: PRMPHeteroSAGE(node_dims, edge_types, prmp_edge_keys, hidden_dim=HIDDEN_DIM)),
    ("C_WideSAGE_Control", lambda: WideHeteroSAGE(node_dims, edge_types, hidden_dim=wide_hidden)),
]

all_results = {}
best_models = {}

for model_name, model_factory in model_configs:
    print(f"\n--- Training: {model_name} ---")
    seed_results = []
    best_model_for_variant = None
    for seed in SEEDS:
        model = model_factory()
        metrics, trained_model = train_model(model, graph, DEVICE, seed, model_name)
        seed_results.append(metrics)
        if best_model_for_variant is None:
            best_model_for_variant = trained_model
        else:
            del trained_model
        gc.collect()
    all_results[model_name] = seed_results
    best_models[model_name] = best_model_for_variant
    rmses = [r["rmse"] for r in seed_results]
    print(f"  Mean RMSE: {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")

## Phase 5-7: Statistical Analysis

Cohen's d effect sizes and cardinality x R² regime analysis using the **full-scale experiment results** from the loaded data.

In [ ]:
# Use the FULL-SCALE results from the loaded data (pre-computed on 200K nodes)
meta = data["metadata"]
gnn_results = meta["gnn_results"]

def compute_cohens_d(results_dict):
    """Pairwise Cohen's d on test RMSE distributions."""
    variant_names = list(results_dict.keys())
    cohens_d = {}
    for v1, v2 in itertools.combinations(variant_names, 2):
        rmse1 = [r["rmse"] for r in results_dict[v1]["per_seed"]]
        rmse2 = [r["rmse"] for r in results_dict[v2]["per_seed"]]
        mean_diff = np.mean(rmse1) - np.mean(rmse2)
        pooled_std = np.sqrt((np.var(rmse1, ddof=1) + np.var(rmse2, ddof=1)) / 2)
        d = float(mean_diff / pooled_std) if pooled_std > 0 else 0.0
        cohens_d[f"{v1}_vs_{v2}"] = round(d, 4)
    return cohens_d

cohens_d = compute_cohens_d(gnn_results)
print("Pairwise Cohen's d (RMSE):")
for pair, d in cohens_d.items():
    print(f"  {pair}: d = {d:.4f}")

# Regime analysis
r2_diag = meta["r2_diagnostic"]
print("\nRegime Analysis (Cardinality x R² interaction):")
regime = meta["regime_analysis"]
print(f"  PRMP relative improvement: {regime['prmp_relative_improvement_rmse']:.4f}")
print(f"  PRMP improves: {regime['prmp_improves']}")
print("\nTop interaction links:")
for link in regime["top_interaction_links"]:
    print(f"  {link['link']}: card_x_r2={link['card_x_r2']:.2f} (card={link['card']:.0f}, r2={link['r2']:.4f})")

## Results Visualization

Comparing full-scale experiment results across all 3 GNN variants.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: RMSE comparison across models ---
ax = axes[0, 0]
model_names = list(gnn_results.keys())
display_names = ["StandardSAGE", "PRMP", "WideSAGE"]
colors = ["#4C72B0", "#DD8452", "#55A868"]
means = [gnn_results[m]["mean_rmse"] for m in model_names]
stds = [gnn_results[m]["std_rmse"] for m in model_names]
bars = ax.bar(display_names, means, yerr=stds, color=colors, capsize=8, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Test RMSE")
ax.set_title("RMSE Comparison (3 seeds, 200K nodes)")
ax.set_ylim(min(means) * 0.95, max(means) * 1.08)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.0003,
            f"{m:.4f}", ha="center", va="bottom", fontsize=9)

# --- Plot 2: Per-seed RMSE ---
ax = axes[0, 1]
for i, (m_name, d_name) in enumerate(zip(model_names, display_names)):
    seed_rmses = [r["rmse"] for r in gnn_results[m_name]["per_seed"]]
    seeds = [r["seed"] for r in gnn_results[m_name]["per_seed"]]
    ax.scatter(seeds, seed_rmses, color=colors[i], label=d_name, s=80, zorder=5)
    ax.plot(seeds, seed_rmses, color=colors[i], alpha=0.3, linestyle="--")
ax.set_xlabel("Seed")
ax.set_ylabel("Test RMSE")
ax.set_title("Per-Seed RMSE Variation")
ax.legend()

# --- Plot 3: Embedding R² (PRMP vs Standard) for top links ---
ax = axes[1, 0]
emb_r2 = meta["embedding_r2"]
links = list(emb_r2["A_StandardSAGE"].keys())
# Pick links with positive R² for readability
valid_links = [l for l in links if emb_r2["A_StandardSAGE"][l]["embedding_r2_mean"] > 0
               and emb_r2["B_PRMP_Full"][l]["embedding_r2_mean"] > 0]
short_labels = [l.split("__")[0][:8] + ".." + l.split("__")[-1][:5] for l in valid_links]
std_r2 = [emb_r2["A_StandardSAGE"][l]["embedding_r2_mean"] for l in valid_links]
prmp_r2_vals = [emb_r2["B_PRMP_Full"][l]["embedding_r2_mean"] for l in valid_links]
x = np.arange(len(valid_links))
w = 0.35
ax.barh(x - w/2, std_r2, w, label="StandardSAGE", color=colors[0], edgecolor="black", linewidth=0.3)
ax.barh(x + w/2, prmp_r2_vals, w, label="PRMP", color=colors[1], edgecolor="black", linewidth=0.3)
ax.set_yticks(x)
ax.set_yticklabels(short_labels, fontsize=7)
ax.set_xlabel("Embedding R²")
ax.set_title("Embedding-Space R² by FK Link")
ax.legend(fontsize=8)

# --- Plot 4: Cardinality x R² interaction ---
ax = axes[1, 1]
interaction = meta["regime_analysis"]["interaction_terms_card_x_r2"]
sorted_items = sorted(interaction.items(), key=lambda x: x[1], reverse=True)
top_items = [(k, v) for k, v in sorted_items if v > 0][:6]
if top_items:
    labels = [k.split("__")[0][:10] + ".." for k, _ in top_items]
    values = [v for _, v in top_items]
    ax.barh(labels, values, color="#8172B3", edgecolor="black", linewidth=0.3)
    ax.set_xlabel("Cardinality x R² Interaction")
    ax.set_title("Top FK Links by Interaction Strength")
else:
    ax.text(0.5, 0.5, "No positive interactions", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("results_summary.png", dpi=100, bbox_inches="tight")
plt.show()
print("Results summary saved to results_summary.png")

# --- Print summary table ---
print("\n" + "="*70)
print("FULL-SCALE EXPERIMENT SUMMARY (rel-avito ad-ctr, 200K AdsInfo nodes)")
print("="*70)
print(f"{'Model':<25} {'RMSE':>10} {'MAE':>10} {'R²':>10} {'Params':>12}")
print("-"*70)
for m_name, d_name in zip(model_names, display_names):
    r = gnn_results[m_name]
    print(f"{d_name:<25} {r['mean_rmse']:.4f}±{r['std_rmse']:.4f} "
          f"{r['mean_mae']:.4f}±{r['std_mae']:.4f} "
          f"{r['mean_r2']:.4f}±{r['std_r2']:.4f} "
          f"{r['param_count']:>10,}")
print("-"*70)
print(f"\nCohen's d (Std vs PRMP): {meta['pairwise_cohens_d']['A_StandardSAGE_vs_B_PRMP_Full']:.4f}")
print(f"PRMP relative improvement: {regime['prmp_relative_improvement_rmse']:.4f} ({regime['prmp_relative_improvement_rmse']*100:.2f}%)")
print(f"PRMP improves over Standard: {regime['prmp_improves']}")